# CP201A Lab 12: Cleaning and Recategorizing Survey Data

**Planning Methods Gateway, Fall 2026 · Wednesday, November 18 (101) and Friday, November 20 (102)**

**What you submit this week:** nothing from this notebook yet. Save your work. Your cleaned notebook becomes Part 1 of **P/NP #11** (due Sunday, November 29), along with two statistical tests.

The lab has two parts:

**Part 1: Clean and recode (Sections 2 to 7).** In this part you will:
- Load our class survey and get it ready to analyze: drop the example and blank rows, give each survey a unique ID, and shorten the column names
- Fix the problems we found on Monday: extra spaces, spelling variants, text in number questions, and ranges that turned into dates
- Recode: turn agree-disagree answers into a dummy and a 1 to 5 score, and make other dummy variables
- Describe your data: counts and shares, means and medians

**Part 2: Your hypotheses (Section 8).** Using the Lab 12 Hypotheses worksheet, write two hypotheses about our survey, code the variables they need, and describe each relationship with a crosstab or a group comparison. Then save a cleaned CSV for Lab 13 and P/NP #11 (Section 9).

Bring the two hypotheses you wrote on November 9 (one outcome, Y, and two explanatory variables, X), and the worksheet from bCourses. Exercises are numbered `# EXERCISE #1`, `# EXERCISE #2`, and so on.

## 0. Getting the data

The class data file, **`CP201A_2026_survey_data.csv`**, needs to sit in the same folder as this notebook. If the cell below can't find it, download the file from the Lab 12 page on bCourses, then in Datahub open this notebook's folder and click the **Upload** button (the up arrow at the top of the file browser) to add it.

A note on the file: we replaced the surveyor last names with a code (S01, S02, and so on) and removed the notes column, so the shared file carries no names.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_FILE = 'CP201A_2026_survey_data.csv'
if os.path.exists(DATA_FILE):
    raw = pd.read_csv(DATA_FILE)
    print(f'Loaded {len(raw)} rows and {raw.shape[1]} columns.')
else:
    print(f'Could not find {DATA_FILE}. Upload it to this folder (see the instructions above), then rerun this cell.')

## 1. Python you'll need today

Four patterns do most of the work in this lab. You've seen all of them; here they are in one place.

| Pattern | What it does | Example |
|---|---|---|
| `df['col'].value_counts(dropna=False)` | Counts every answer, including blanks (`NaN`) | How many people chose each answer? |
| `df['col'].map({...})` | Swaps each value for a new one using a dictionary | `'Rent'` becomes `1`, `'Own'` becomes `0` |
| `df['col'].apply(my_function)` | Runs your own function on every value | Turn `'2 years 6 months'` into `2.5` |
| `pd.crosstab(df['a'], df['b'], normalize='index')` | A two-way table with row percents | Agree vs. disagree, by renters and owners |

And a **function** is a recipe you write once and reuse. The `def` line names it and its input; `return` gives back the result:

In [ ]:
def double(x):
    return x * 2

double(21)

# Part 1: Clean and recode

## 2. First look and basic housekeeping

### 2.1 Look before you change anything

Always start by looking. `.head()` shows the first rows; `.info()` shows each column's name, how many cells have something in them, and its data type. A number question showing `object` (text) instead of `float64` is a sign that some answers aren't numbers yet.

In [ ]:
raw.head(3)

In [ ]:
raw.info()

### 2.2 Drop the example row and blank rows

The data entry sheet started with an example row, and a few rows are blank or nearly blank. We keep the original (`raw`) untouched and make a working copy (`df`).

In [ ]:
df = raw.copy()

# The example row has 'Example' in the first column
df = df[df['Unnamed: 0'] != 'Example'].drop(columns=['Unnamed: 0'])

# Drop rows with no city (blank or nearly blank rows)
before = len(df)
df = df[df['City'].notna()].reset_index(drop=True)
print(f'Dropped {before - len(df)} blank rows. {len(df)} surveys remain.')

### 2.3 Give every survey a unique ID

Survey numbers use the surveyor's initials, so three different students each have an `MK-1`. The survey number alone doesn't identify a survey. Combining the surveyor code with the survey number does.

In [ ]:
print('Duplicate survey numbers:', df['Survey Number'].duplicated().sum())

df['survey_id'] = df['Surveyor'] + '_' + df['Survey Number'].astype(str)
print('Duplicate survey_id values:', df['survey_id'].duplicated().sum())

### 2.4 Shorter column names

Long question text is hard to type. We rename every column once, here. **This dictionary is the start of your codebook:** the short name on the right, the question on the left. Copy it into your codebook document if you keep one separately.

In [ ]:
rename_map = {
    'Surveyor': 'surveyor',
    'Survey Number': 'survey_num',
    'Day of the Week': 'day',
    'Time of Day': 'time',
    'Interviewer or self-administered': 'mode',
    'City': 'city',
    'Neighborhood': 'neighborhood',
    'Business or organization': 'place_kind',
    'Survey Language': 'language',
    'Interview?': 'interview_interest',
    'What best describes your role here?': 'role',                                         # Q1
    'What type of place is this?': 'place_type',                                           # Q2
    'About how many years has this business or organization been at this location?': 'years_at_location',  # Q3
    'About how many years have you worked or volunteered here?': 'years_worked',           # Q4
    'Including you, about how many people work here?': 'num_workers',                      # Q5
    'Does this business or organization own or rent its space?': 'space_tenure',          # Q6
    'Do you live in this neighborhood?': 'lives_here',                                     # Q7
    'How many years have you lived here?': 'years_lived',                                  # Q7, if yes
    'Do you own or rent your home?': 'home_tenure',                                        # Q8
    'On a typical workday, about how many minutes does it take you to get here?': 'commute_min',  # Q9
    'How do you usually get here?': 'travel_mode',                                         # Q10
    'This neighborhood has changed a lot in the past five years.': 'changed_a_lot',        # 2a
    'Changes in this neighborhood in the past five years have been good for long-time residents.': 'good_for_longtime',  # 2b
    'Changes in this neighborhood in the past five years have been good for businesses and organizations like this one.': 'good_for_business',  # 2c
    'Long-time residents can afford to stay in this neighborhood.': 'afford_to_stay',      # 2d
    'I support building more housing in this neighborhood.': 'support_housing',            # 2e
    'I feel connected to the people and community in this neighborhood.': 'connected',     # 2f
    'I feel safe walking alone at night in this neighborhood.': 'safe_at_night',           # 2g
    'The number of customers or clients who come here': 'p12_customers',                  # 3a
    "This place's sales, revenue, or funding": 'p12_sales',                                # 3b
    "Rent for this business or organization's space": 'p12_space_rent',                   # 3c
    'Other costs of running this place (supplies, wages, utilities)': 'p12_other_costs',   # 3d
    'Your own rent or mortgage payment': 'p12_own_housing',                                # 3e
    'Prices of everyday goods and services in this neighborhood': 'p12_prices',            # 3f
    'What is the biggest change you have noticed in this neighborhood in the past five years?': 'biggest_change',  # Q11
    'What is the biggest challenge for businesses and organizations in this neighborhood right now?': 'biggest_challenge',  # Q12
    'How old are you?': 'age',
    'What is your current gender?': 'gender',
    'Which language(s) do you use most at work? Check all that apply.': 'work_languages',
    'What is your race or ethnicity?': 'race_eth',
}
df = df.rename(columns=rename_map)

# Check: any column we missed still has its long name
print([c for c in df.columns if c not in rename_map.values() and c != 'survey_id'])

## 3. Cleaning text answers

### 3.1 Extra spaces

`'Oakland'` and `'Oakland '` (with a space at the end) look the same to us and different to Python. Python's `.strip()` removes spaces at the start and end of a piece of text. We write a tiny function that strips text and leaves numbers and blanks alone, then use a `for` loop to apply it to every column at once.

In [ ]:
print('Before:', df['city'].unique())

def strip_spaces(v):
    return v.strip() if isinstance(v, str) else v   # only text has spaces to strip

for col in df.columns:
    df[col] = df[col].map(strip_spaces)

print('After: ', df['city'].unique())

### 3.2 Spelling and capitalization

Next, spelling variants. Look first with `value_counts()`, decide what each variant should become, then fix it with `.replace()`. Only list the values you want to change; everything else stays the same.

In [ ]:
df['place_kind'].value_counts(dropna=False)

In [ ]:
df['place_kind'] = df['place_kind'].replace({'Buisness': 'Business', 'non-profit': 'Organization'})
df['place_kind'].value_counts(dropna=False)

Notice the answers that remain, like a program name typed into this column. Should they become `'Organization'`? That's a judgment call. Make it, and write it in your codebook.

Neighborhood names need the same treatment. Here we put them into a consistent case with `.str.title()`, which fixes `'Iron triangle'` vs. `'Iron Triangle'`:

In [ ]:
df['neighborhood'] = df['neighborhood'].str.title()
df.groupby('city')['neighborhood'].value_counts()

In [ ]:
# EXERCISE #1: Some neighborhood names still need decisions. In Berkeley, 'Downtown' and
# 'Downtown Berkeley' are the same place, and 'South' probably means 'South Berkeley'.
# Use .replace() with a dictionary to merge the variants you're confident about, then rerun
# the groupby above to check. Write one sentence (as a # comment) about a variant you chose NOT to merge, and why.


## 4. Cleaning number questions

### 4.1 What's wrong with the numbers?

Five questions ask for numbers: years at this location (Q3), years worked here (Q4), people working here (Q5), years lived here (Q7), and commute minutes (Q9). Because people wrote things like `'6 months'`, `'2 years 8 months'`, and `"Don't know"`, Python reads these columns as text. Look at a sample:

In [ ]:
df['years_worked'].dropna().sample(15, random_state=1).tolist()

And on Monday we saw a worse problem. When someone typed a range like `10-15`, the spreadsheet decided it was a date (October 15) and stored it that way. Let's find those cells:

In [ ]:
number_cols = ['years_at_location', 'years_worked', 'num_workers', 'years_lived', 'commute_min']

def looks_like_date(x):
    # A date looks like 2026-10-15 or 10/15/2026, depending on how the sheet was exported
    return isinstance(x, str) and (re.match(r'^\d{4}-\d{2}-\d{2}', x) is not None
                                   or re.match(r'^\d{1,2}/\d{1,2}/\d{2,4}', x) is not None)

for col in number_cols:
    dates = df.loc[df[col].apply(looks_like_date), ['survey_id', col]]
    if len(dates) > 0:
        print(col)
        print(dates.to_string(index=False), '\n')

A date like `2026-10-15` was probably a range of 10 to 15, but we can't be sure from the spreadsheet. The safe rule is to set these cells to missing (`NaN`) and note them; the person who entered the survey can check the paper copy. **Our rule: a date in a number question becomes `NaN`.** Write it in your codebook.

### 4.2 A function that turns answers into numbers

This function handles the patterns in our data. Read the comments: each step is one rule, and each rule is a decision you could make differently.

In [ ]:
def to_years(x):
    '''Turn an answer like '2 years 6 months' into a number of years (2.5).
    Returns NaN for blanks, "don't know," and dates.'''
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):          # already a number
        return float(x)
    s = str(x).lower()
    if looks_like_date(s):                   # a range the spreadsheet turned into a date
        return np.nan
    if 'know' in s or s in ['n/a', 'na', 'skipped', 'did not answer']:
        return np.nan
    m = re.match(r'^\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*$', s)  # a range like '300-400'
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2          # rule: use the midpoint
    total = 0
    found = False
    for number, unit in re.findall(r'(\d+(?:\.\d+)?)\s*(year|yr|month|mo|week|day)?', s):
        found = True
        n = float(number)
        if unit.startswith('mo'):
            total += n / 12
        elif unit == 'week':
            total += n / 52
        elif unit == 'day':
            total += n / 365
        else:                                # 'years' or no unit
            total += n
    return total if found else np.nan

# Try it on a few answers before using it on the whole column
for test in ['6 months', '2 years 8 months', '10', "Don't know", '300-400', '2026-10-15 00:00:00']:
    print(f'{test!r:>25} -> {to_years(test)}')

In [ ]:
df['years_worked_num'] = df['years_worked'].apply(to_years)
df['years_at_location_num'] = df['years_at_location'].apply(to_years)
df['years_lived_num'] = df['years_lived'].apply(to_years)
df['num_workers_num'] = df['num_workers'].apply(to_years)   # same rules work for counts

# Always compare the original and the cleaned version side by side
df[['years_worked', 'years_worked_num']].dropna().sample(10, random_state=2)

Commute minutes need a slightly different function, since some people wrote hours:

In [ ]:
def to_minutes(x):
    '''Turn '1 hour' or '45 minutes' into minutes. NaN for blanks, "don't know," and dates.'''
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).lower()
    if looks_like_date(s) or 'know' in s or 'skip' in s:
        return np.nan
    m = re.match(r'^\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)', s)
    if m:
        value = (float(m.group(1)) + float(m.group(2))) / 2
    else:
        nums = re.findall(r'\d+(?:\.\d+)?', s)
        if not nums:
            return np.nan
        value = float(nums[0])
    return value * 60 if ('hour' in s or 'hr' in s) else value

df['commute_num'] = df['commute_min'].apply(to_minutes)
df[['commute_min', 'commute_num']].dropna().sample(10, random_state=3)

### 4.3 Describe the numbers, and check the extremes

`.describe()` gives the count, mean, standard deviation, minimum, quartiles (25%, 50% = the median, 75%), and maximum. **Always look at the minimum and maximum** before you trust a mean.

In [ ]:
df[['years_worked_num', 'years_at_location_num', 'years_lived_num', 'num_workers_num', 'commute_num']].describe()

In [ ]:
# The largest values: real, or entry errors?
df.sort_values('num_workers_num', ascending=False)[['survey_id', 'place_type', 'num_workers', 'num_workers_num']].head(8)

Some large values are real (a library or a big employer), and some may be errors. You don't delete real values because they're inconvenient. You report the **median** alongside the mean when a few large values pull the mean up. Compare them:

In [ ]:
for col in ['years_worked_num', 'years_at_location_num', 'num_workers_num', 'commute_num']:
    print(f'{col:>22}: mean {df[col].mean():6.1f}   median {df[col].median():6.1f}   n = {df[col].notna().sum()}')

In [ ]:
# EXERCISE #2: Make a histogram of years_worked_num with df['years_worked_num'].plot.hist(bins=20).
# Give it a title and an x-axis label (plt.title(...), plt.xlabel(...)).
# Then, in a # comment, say in one sentence whether the mean or the median is the better summary, and why.


## 5. Categories: counts, shares, and crosstabs

For a category, start with counts and shares. `normalize=True` turns counts into shares. **Always say what the share is out of** (all respondents? only those who answered?).

In [ ]:
print(df['role'].value_counts(dropna=False), '\n')
print((df['role'].value_counts(normalize=True) * 100).round(1))   # share of those who answered

A **crosstab** puts two variables together. `normalize='index'` gives row percents: each row adds to 100. Here: of the people who live in the neighborhood, what share own or rent their home, and the same for people who don't?

In [ ]:
pd.crosstab(df['lives_here'], df['home_tenure'], normalize='index').round(3) * 100

In [ ]:
pd.crosstab(df['lives_here'], df['home_tenure'], margins=True)   # counts, with totals

## 6. Recoding the agree-disagree grid

### 6.1 Put the categories in order

Our Section 2 statements use the same scale. Telling pandas the order makes tables and charts read from strongly disagree to strongly agree instead of alphabetically.

In [ ]:
grid2 = ['changed_a_lot', 'good_for_longtime', 'good_for_business', 'afford_to_stay',
         'support_housing', 'connected', 'safe_at_night']

agree_order = ['Strongly disagree', 'Disagree', 'Neutral', 'Agree', 'Strongly agree', "Don't know"]
agree_type = pd.CategoricalDtype(agree_order, ordered=True)

df['support_housing'].astype(agree_type).value_counts(sort=False, dropna=False)

### 6.2 Two versions of each statement: a dummy and a score

We make two new columns for each statement:

- **`_agree_dv`**, a dummy: 1 if agree or strongly agree, 0 if neutral, disagree, or strongly disagree, and `NaN` for don't know and blanks. Use it to compare **shares** between groups (a Z-test of proportions).
- **`_score`**, from 1 (strongly disagree) to 5 (strongly agree), `NaN` for don't know. Use it to compare **averages** (a t-test).

Counting neutral as 0 is a decision. It treats "not agreeing" as the comparison group. You could instead set neutral to `NaN` and compare only people who took a side. Whatever you choose, say so in your Data Note.

In [ ]:
agree_dummy = {'Strongly agree': 1, 'Agree': 1, 'Neutral': 0, 'Disagree': 0,
               'Strongly disagree': 0, "Don't know": np.nan}
agree_score = {'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4,
               'Strongly agree': 5, "Don't know": np.nan}

for col in grid2:
    df[col + '_agree_dv'] = df[col].map(agree_dummy)
    df[col + '_score'] = df[col].map(agree_score)

# Check one: every original answer should line up with its new codes
df[['support_housing', 'support_housing_agree_dv', 'support_housing_score']].value_counts(dropna=False).sort_index()

In [ ]:
# Share who agree with each statement, among those who gave an opinion (don't know and blanks left out)
summary = pd.DataFrame({
    'share_agree_%': [df[c + '_agree_dv'].mean() * 100 for c in grid2],
    'mean_score_1to5': [df[c + '_score'].mean() for c in grid2],
    'n': [df[c + '_agree_dv'].notna().sum() for c in grid2],
}, index=grid2).round(1)
summary.sort_values('share_agree_%', ascending=False)

### 6.3 The past-12-months grid

Section 3 uses a different scale, from decreased a lot to increased a lot. The same idea applies: a dummy for "increased" and a score from 1 to 5.

In [ ]:
grid3 = ['p12_customers', 'p12_sales', 'p12_space_rent', 'p12_other_costs', 'p12_own_housing', 'p12_prices']

increased_dummy = {'Increased a lot': 1, 'Increased a little': 1, 'Stayed about the same': 0,
                   'Decreased a little': 0, 'Decreased a lot': 0, "Don't know or doesn't apply": np.nan}
change_score = {'Decreased a lot': 1, 'Decreased a little': 2, 'Stayed about the same': 3,
                'Increased a little': 4, 'Increased a lot': 5, "Don't know or doesn't apply": np.nan}

for col in grid3:
    df[col + '_increased_dv'] = df[col].map(increased_dummy)
    df[col + '_score'] = df[col].map(change_score)

pd.DataFrame({'share_increased_%': [df[c + '_increased_dv'].mean() * 100 for c in grid3],
              'n': [df[c + '_increased_dv'].notna().sum() for c in grid3]}, index=grid3).round(1)

## 7. More dummy variables

Any category can become a dummy. Decide what counts as 1, what counts as 0, and what becomes `NaN`. Then **check** with `value_counts(dropna=False)` on the original and the new column together.

In [ ]:
# Lives in the neighborhood (Q7)
df['lives_here_dv'] = df['lives_here'].map({'Yes': 1, 'No': 0})

# Rents their home (Q8). Decision: 'Other' (for example, living with family) and 'Skipped' become NaN,
# so this dummy compares renters with owners only.
df['rent_home_dv'] = df['home_tenure'].map({'Rent': 1, 'Own': 0})

# The place rents its space (Q6); "Don't know" becomes NaN
df['space_rented_dv'] = df['space_tenure'].map({'Rent': 1, 'Own': 0})

# Owner (Q1): 1 for owners, 0 for everyone else who answered
df['owner_dv'] = np.where(df['role'].isna(), np.nan, (df['role'] == 'Owner').astype(float))

# Self-administered survey (mode), useful as a covariate
df['self_admin_dv'] = df['mode'].map({'Self-administered': 1, 'Interviewer': 0})

df[['home_tenure', 'rent_home_dv']].value_counts(dropna=False)

**Check-all-that-apply answers.** Travel mode (Q10) asked for one answer, but some people checked several, so cells read `'Walk, Public transit'`. `.str.contains()` asks whether a mode appears anywhere in the answer:

In [ ]:
df['drives_alone_dv'] = np.where(df['travel_mode'].isna(), np.nan,
                                df['travel_mode'].str.contains('Drive alone').astype(float))
df['transit_dv'] = np.where(df['travel_mode'].isna(), np.nan,
                            df['travel_mode'].str.contains('Public transit').astype(float))
df[['travel_mode', 'drives_alone_dv', 'transit_dv']].value_counts(dropna=False).head(10)

In [ ]:
# EXERCISE #3: Make a dummy for a variable in YOUR hypothesis that we haven't coded yet
# (for example, a race or ethnicity category, an age group, a type of place, or a city).
# Name it with _dv at the end, decide what becomes NaN, and check it with value_counts(dropna=False)
# on the original and the new column together.


# Part 2: Your hypotheses

Have your Lab 12 Hypotheses worksheet open. Fill in the Y and X lines first, then come back here.

## 8. Describe your own hypothesis

Now use what you've built on the hypothesis you wrote on November 9. Two patterns cover most cases:

- **Y is a dummy or a category, X is a group:** a crosstab with row percents.
- **Y is a number (or a 1 to 5 score), X is a group:** `.groupby()` with the mean and median.

Here's an example of each. On Wednesday's lecture and in Lab 13 we'll test whether these differences are statistically significant.

In [ ]:
# Example 1: do residents and non-residents differ in agreeing the neighborhood has changed a lot?
pd.crosstab(df['lives_here'], df['changed_a_lot_agree_dv'], normalize='index').round(3) * 100

In [ ]:
# Example 2: do renters and owners differ in their average support-for-housing score?
df.groupby('home_tenure')['support_housing_score'].agg(['mean', 'median', 'count']).round(2)

In [ ]:
# EXERCISE #4: Describe the relationship in your first hypothesis.
# Write your hypothesis as a # comment first (Y, X, and the direction you expect).
# Then make a crosstab (with normalize='index') or a groupby (with mean, median, and count).


In [ ]:
# EXERCISE #5: Do the same for your second hypothesis.


In [ ]:
# EXERCISE #6: In two or three sentences (as # comments), describe what you found for one hypothesis,
# the way you would in your final memo: the shares or means for each group, how many people are in each group,
# and what's left out (don't know, blanks, 'other'). Don't claim a difference is significant yet; that's next week.


## 9. Save your cleaned data

Save the cleaned data as a CSV. You'll read it back in for Lab 13 and P/NP #11, so you don't have to clean everything again. Then download it as a backup (right-click the file in the Datahub file browser, then **Download**).

In [ ]:
df.to_csv('survey_cleaned_2026.csv', index=False)
print(f'Saved {len(df)} rows and {df.shape[1]} columns to survey_cleaned_2026.csv')

## Before you leave

- Your notebook runs top to bottom: **Kernel > Restart Kernel and Run All Cells**.
- Exercises #1 to #6 are done, or you've noted where you're stuck.
- Your codebook lists every new variable you made, how it's coded, and what became `NaN`.
- `survey_cleaned_2026.csv` is saved, and you've downloaded a copy.

This notebook is Part 1 of P/NP #11 (due Sunday, November 29).